# Principal Factor: Modeling Proposal

This notebook contains the modeling design, four benchmark experiments, directly relevant IM3 data sources, and training recommendations. Definitions of the 15 core features and the H1–H13 hard constraints are retained in `01_us_site_inventory_and_feature_schema.ipynb`. This is a proposal and does not read local data files or report fitted-model results.


# Part 3: Four Comparisons of Single-Model Fusion and Factor-Wise Late Fusion

## 3.1 Comparison design

This section compares two training architectures only: (1) **single-model fusion**, in which all 15 core features enter one model; and (2) **factor-wise training followed by late fusion**, in which five submodels—power, water/cooling, network, land/construction, and cost/market—are trained first and an out-of-fold meta-model combines their predictions. Each architecture is evaluated as both regression and ordinal multiclass classification, producing four primary experiments.

Both architectures apply the H1–H13 hard constraints first. IT load and PUE appear in both the power and water-related feature groups because they determine power demand and cooling demand through different physical processes. This is legitimate reuse of project parameters, not duplicated score construction.

## 3.2 Five submodels required for factor-wise late fusion

### 3.2.1 Power submodel

| Measured meaning | Model feature | Unit or encoding | IM3 alignment | Availability |
|---|---|---|---|---|
| Rated power required by servers and other IT equipment | `data_center_it_power_mw` | MW | CERF-DC facility-energy and electricity-cost input | A: project plan, utility interconnection request, or OSTI project field |
| Ratio of total facility energy to IT-equipment energy | `data_center_pue` | ratio | CERF-DC converts IT load to total facility load | B: project design, energy disclosure, or operating record; do not fabricate when missing |
| Distance to the nearest substation | `interconnection_distance_km` | km | IM3 2 km interconnection threshold and CERF-DC interconnection cost | A: calculated from HIFLD/GeoPlatform substation coordinates; screen out values `>2 km` first |
| Rated voltage class of nearby transmission lines | `transmission_voltage_class` | kV or voltage class | IM3 Atlas retains HIFLD `VOLT_CLASS` | A: HIFLD/GeoPlatform transmission-line attribute |

**Targets:** Regression uses the continuous subscore `s_power`; ordinal classification uses `y_power ∈ {1,…,K}`. Both should be labeled from realized interconnection outcomes, utility serviceability opinions, or consistent expert review. Electricity price is an economic outcome and belongs in the cost model so that physical serviceability is not mixed with price.

### 3.2.2 Water and cooling submodel

| Measured meaning | Model feature | Unit or encoding | IM3 alignment | Availability |
|---|---|---|---|---|
| Rated power required by servers and other IT equipment | `data_center_it_power_mw` | MW | CERF-DC cooling demand varies with IT load | A: project plan, utility interconnection request, or OSTI project field |
| Ratio of total facility energy to IT-equipment energy | `data_center_pue` | ratio | CERF-DC facility and cooling-energy calculation | B: project design, energy disclosure, or operating record; do not fabricate when missing |
| Fraction of project cooling load served by water-based cooling | `water_cooling_fraction` | 0–1 | CERF-DC water withdrawal, consumption, and cooling-energy input | B: project design, water permit, or environmental-review filing |
| Minimum distance to a public water-supply service area | `municipal_water_distance_km` | km; 0 inside a service area | IM3 uses USGS WSA and a 5 km threshold | A: calculated from USGS WSA v1 boundaries; screen out values `>5 km` first |

**Targets:** Regression uses `s_water`; ordinal classification uses `y_water ∈ {1,…,K}`. Labels should reflect utility serviceability, water permits, or a consistent expert assessment of the ability to meet project cooling demand. Annual withdrawal and consumption calculated from these inputs are explanatory outputs and must not re-enter `X`.

### 3.2.3 Network submodel

| Measured meaning | Model feature | Unit or encoding | IM3 alignment | Availability |
|---|---|---|---|---|
| Number of providers offering symmetric 1 Gbps business broadband near the site | `fiber_provider_count_1gbps` | provider count | IM3 Atlas aggregates FCC business-service records by H3 cell | A: FCC Broadband Data Collection |
| Minimum distance to a high-speed commercial fiber service area | `high_speed_fiber_distance_km` | km; 0 inside a service area | IM3 H9 uses 2 km as the high-speed fiber-access threshold | A: calculated from FCC service-area boundaries; screen out values `>2 km` first |

**Targets:** Regression uses `s_network`; ordinal classification uses `y_network ∈ {1,…,K}`. Labels should reflect reachable provider count, confirmed service, or expert review. Because this submodel has very few features, prioritize GAMs, tree models, and ordinal logistic regression rather than forcing a deep network.

### 3.2.4 Land and construction submodel

| Measured meaning | Model feature | Unit or encoding | IM3 alignment | Availability |
|---|---|---|---|---|
| Total area required by the proposed campus | `campus_size_square_ft` | ft² | CERF-DC determines the number of required contiguous cells | A: project plan, permit filing, or OSTI project field |
| Contiguous buildable land remaining after all hard constraints | `feasible_contiguous_area_sqft` | ft² | CERF-DC selects locations whose contiguous feasible area meets campus requirements | A/B: connected-component calculation from the IM3 suitability raster |
| Maximum surface slope within the candidate parcel | `max_slope_pct` | % | IM3 H3 uses 16% as an exclusion threshold | A: calculated from USGS 3DEP elevation data; screen out values `>16%` first |
| Distance to the nearest developed land | `developed_land_distance_km` | km | IM3 H13 requires a location within 0.8 km of developed land | A: calculated from NLCD developed land; screen out values `>0.8 km` first |

**Targets:** Regression uses `s_land`; ordinal classification uses `y_land ∈ {1,…,K}`. Labels should reflect the buildable-area margin relative to project size, slope, and actual planning or permitting outcomes. Airport, waterbody, flood, protected-land, and military overlap variables are constant-pass after hard screening and do not become ordinary scoring features.

### 3.2.5 Cost and market submodel

| Measured meaning | Model feature | Unit or encoding | IM3 alignment | Availability |
|---|---|---|---|---|
| Land acquisition or assessed cost per unit area | `land_cost_per_sqft` | USD/ft² | CERF-DC locational-cost land input | B: IM3 raster, county parcel assessment, or transaction record |
| Electricity charge per unit consumed | `electricity_rate_per_kwh` | USD/kWh | CERF-DC annual electricity-cost input | A/B: EIA or utility tariff; normalize to one base year |
| Local property-tax rate applicable to servers and equipment | `personal_property_tax_rate` | decimal | CERF-DC equipment property-tax input | B: published state and local tax rates |
| Local real-property tax rate applicable to land and buildings | `real_property_tax_rate` | decimal | CERF-DC land and building property-tax input | B: published state and local tax rates |
| Sales-tax rate applicable to equipment or electricity | `sales_tax_rate` | decimal | CERF-DC equipment and energy sales-tax input | B: published state and local tax rates |
| Distance to the nearest Census market or population center | `market_distance_km` | km | CERF-DC market gravity uses distance to the nearest market | B: IM3 market raster or Census market-center coordinates |

**Targets:** Regression uses `s_cost_market`; ordinal classification uses `y_cost_market ∈ {1,…,K}`. Labels should use verifiable costs normalized to one base year and market distance. `total_cost_million_usd`, normalized cost, and gravity scores are calculated outcomes; they may serve as baselines or auxiliary labels but cannot feed back as features.

## 3.3 Training the two architectures

All four experiments must use identical spatial splits. Inner spatial cross-validation selects the algorithm, hyperparameters, and number of grades `K`; unseen outer regions support unbiased comparison; a locked geographic test set is used once at the end.

1. **Single-model fusion—regression:** Enter all 15 core features into one regressor to predict `y_score`.
2. **Single-model fusion—ordinal classification:** Enter the same 15 features into one ordinal classifier to predict `y_grade` and grade probabilities.
3. **Factor-wise late fusion—regression:** The five factor models predict `s_power`, `s_water`, `s_network`, `s_land`, and `s_cost_market`; the meta-model reads five OOF (out-of-fold) continuous subscores and predicts `y_score`.
4. **Factor-wise late fusion—ordinal classification:** The five factor models predict grade probabilities; the meta-model reads `5×K` OOF probabilities and predicts `y_grade`.
5. **Leakage control:** A late-fusion model must never read base-model predictions on the samples used to fit those base models. Grade thresholds, imputers, and scalers must also be fitted within each training fold.
6. **Hard constraints first:** Any site triggering H1–H13 is excluded directly and cannot be rescued by a high regression score or recommendation probability.

## 3.4 Two supervised targets

| Task | Final label | Interpretation | Primary metric | Secondary metrics |
|---|---|---|---|---|
| Regression | `y_score` | Continuous recommendation score; higher is better | MAE | RMSE, R², Spearman correlation |
| Ordinal multiclass classification | `y_grade` | Grade 1 is best and grade K is worst; compare K=3, 4, and 5 | QWK and grade MAE | Macro-F1, within-one-grade accuracy, probability calibration |

`y_score` should preferably come from realized project outcomes or a consistent expert continuous score; existing composite scores such as CERF-DC are weak-label baselines only. `y_grade` should preferably be labeled independently. If it is created by thresholding `y_score`, label the process “score discretization,” estimate cut points only within training folds, and do not claim that the two tasks independently validate each other.

## 3.5 Candidate machine-learning and deep-learning models

| Task | Model class | Candidate models | Rationale and comparison focus |
|---|---|---|---|
| Regression | Linear/interpretable ML | Elastic Net, GAM | Stable baselines for near-linear and monotonic relationships; GAM displays feature-response curves |
| Regression | Kernel method | SVR | Appropriate for nonlinear, small-to-medium tabular datasets; training cost grows with sample size |
| Regression | Bagging trees | Random Forest, Extra Trees | Robust to nonlinearities and interactions with limited preprocessing |
| Regression | Boosted trees | XGBoost, LightGBM, CatBoost | Priority candidates for tabular data; compare accuracy, missing-value handling, and stability |
| Regression | Deep learning | MLP, TabNet, FT-Transformer | Test whether larger samples and complex interactions yield gains; may underperform trees on small data |
| Ordinal classification | Interpretable ML | Ordinal logistic regression | Explicitly preserves grade order and provides the lowest-complexity baseline |
| Ordinal classification | Bagging trees | Random Forest, Extra Trees | Nonlinear multiclass baselines; inspect long-distance grade errors |
| Ordinal classification | Ordinal boosting | XGBoost, LightGBM, CatBoost with cumulative thresholds | Learn `P(y≤k)` and exploit grade order on tabular data |
| Ordinal classification | Ordinal deep learning | CORAL/CORN-MLP, CORAL/CORN-TabNet, CORAL/CORN-FT-Transformer | Ordinal losses penalize errors spanning multiple grades |

Deep learning enters formal comparison only when sample size and label quality are adequate. The network submodel has few features and does not require deep learning; its principal candidates are GAMs, tree models, and ordinal logistic regression.

## 3.6 Candidate meta-models for factor-wise late fusion

| Fusion task | Input | Candidate models |
|---|---|---|
| Regression late fusion | Five OOF continuous subscores | Ridge, Elastic Net, LightGBM, CatBoost, small MLP |
| Ordinal late fusion | `5×K` OOF grade probabilities | Ordinal logistic regression, cumulative-threshold LightGBM/CatBoost, CORAL-MLP |

## 3.7 Four primary benchmark experiments

| ID | Training architecture | Task | Input and output | Core comparison purpose |
|---|---|---|---|---|
| E1 | Single-model fusion | Regression | 15 features → `y_score` | Test whether one model can learn cross-factor relationships directly |
| E2 | Factor-wise late fusion | Regression | Five factor subscores → meta-model → `y_score` | Compare modular regression with E1 on accuracy and interpretability |
| E3 | Single-model fusion | Ordinal multiclass classification | 15 features → `y_grade` and probabilities | Test whether one model can learn recommendation grades directly |
| E4 | Factor-wise late fusion | Ordinal multiclass classification | `5×K` factor probabilities → meta-model → `y_grade` | Compare modular classification with E3 on grade-error reduction |

The experiments form a strict `2 (training architectures) × 2 (task formulations)` comparison. The IM3/CERF-DC rule score remains a shared external reference baseline and does not count as a fifth primary experiment. Model selection must not rely on one random split. Use outer-spatial-test MAE for regression and QWK plus grade MAE for ordinal classification; report means, standard deviations, and locked-test results. Publish the best regression and ordinal models separately, and use SHAP, GAM curves, or permutation importance to test whether their reasoning is consistent with real constraints.


# Part 4: Directly Relevant IM3 Data Sources

| Data source | Content used in this proposal | Availability | Link |
|---|---|---|---|
| IM3 Open Source Data Center Atlas | Existing locations, FCC business-fiber provider counts, municipal water service areas, and transmission voltage | Public; location GPKG is included locally | https://github.com/IMMM-SFA/datacenter-atlas |
| Atlas data-preparation notebook | Symmetric 1 Gbps business-fiber filter, H3 provider counts, USGS `WSA_NAME`, and HIFLD `VOLT_CLASS` | Public and reproducible code | https://github.com/IMMM-SFA/datacenter-atlas/blob/main/datacenters.ipynb |
| CERF Data Centers | Calculation logic for land, electricity price, taxes, interconnection distance, PUE, cooling fraction, market distance, and contiguous area | Public code; some input rasters require separate acquisition | https://github.com/IMMM-SFA/cerf_data_centers |
| IM3 Projected U.S. Data Center Locations | Campus area, IT MW, cooling fraction, and derived outcomes; scenario labels and subjective weights are not adopted | Public OSTI/DOI download | https://www.osti.gov/biblio/2571680 |
| FCC National Broadband Map | Commercial high-speed fiber service and provider counts | Public nationwide; requires bulk spatial aggregation | https://broadbandmap.fcc.gov/home |
| USGS Public Supply Service Areas | Municipal water coverage and distance | Public nationwide; preserve service-area vintage | https://doi.org/10.5066/P9I22Z24 |
| HIFLD/GeoPlatform | Substation distance, transmission lines, and voltage classes | Public nationwide; distance does not measure spare capacity | https://hifld-geoplatform.hub.arcgis.com/ |
| IM3 suitability exclusions | Airports, waterbodies, slope, sinkholes, floods, protected land, transportation, military areas, and NLCD developed land | Mostly public nationwide layers; reproject and overlay following the OSTI method | https://www.osti.gov/biblio/2571680 |

Expanded sources in the earlier notebook—Server Country, PeeringDB, OSM, EIA/eGRID, USGS hydrology and water quality, SSURGO, community, and workforce data—are excluded from the first core combination because they are not required inputs to the current IM3/CERF-DC siting calculation.


# Part 5: Minimal Training Plan for Regression and Ordinal Multiclass Classification

1. Create a common spatial unit for existing and candidate sites, retain the 15 observable features defined above, and apply H1–H13 hard screening first.
2. Establish two labels: continuous recommendation score `y_score` and ordered grade `y_grade`; treat `K=3/4/5` as a hyperparameter.
3. Run four primary experiments: single-model regression, factor-wise late-fusion regression, single-model ordinal classification, and factor-wise late-fusion ordinal classification. Compare suitable machine-learning and deep-learning candidates within each experiment.
4. For regression, compare Elastic Net, GAM, SVR, Random Forest, Extra Trees, XGBoost, LightGBM, CatBoost, MLP, TabNet, and FT-Transformer.
5. For ordinal classification, compare ordinal logistic regression, Random Forest, Extra Trees, cumulative-threshold XGBoost/LightGBM/CatBoost, and MLP, TabNet, and FT-Transformer with CORAL/CORN losses.
6. Use nested spatial cross-validation to select the algorithm, hyperparameters, and `K`, and report final performance only once on the locked geographic test set.

### Final recommendations

- Publish both a continuous recommendation score and an ordinal grade, selecting the best-performing model for each without assuming that the same algorithm will win.
- Prioritize LightGBM, CatBoost, GAM, and ordinal logistic regression for tabular data; retain deep learning as a formal comparator that may exploit complex interactions when sample size is sufficient.
- Single-model fusion learns relationships among all 15 features directly. Factor-wise late fusion preserves explanations for the five factors. Compare both under identical spatial splits.
- Consider the modeling approach credible only if it consistently outperforms the IM3/CERF-DC rule baseline on outer spatial tests and its reasoning agrees with real constraints.
- Keep other obtainable but presently low-value features in a deferred candidate list rather than the first training dataset.
